In [ ]:
# KALMAN OPEN REVALIDATION — FROZEN VALIDATION GATE v1.9
# ONE CELL / research-only / no production, broker, or Neon writes

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json, math, hashlib
import numpy as np
import pandas as pd

ROOT=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results/open_revalidation_v1")
AUDIT=ROOT/"open_revalidation_trade_audit.parquet"
OUT=ROOT/"validation_v1_9"
OUT.mkdir(parents=True,exist_ok=True)
SEED=20260924
N_BOOT=10000
REQUIRED=["entry_price_iex","fixed4_exit_price_iex","prev_close_price_iex","open_0_price_iex","open_5_price_iex","open_15_price_iex"]

assert AUDIT.exists(), AUDIT
d=pd.read_parquet(AUDIT).copy()
print("[AUDIT]",AUDIT,"rows=",len(d),"cols=",len(d.columns))

# immutable source fingerprint
src_hash=hashlib.sha256(pd.util.hash_pandas_object(d,index=True).values.tobytes()).hexdigest()
missing={c:int(d[c].isna().sum()) if c in d else len(d) for c in REQUIRED}
ready=d[REQUIRED].notna().all(axis=1) if all(c in d for c in REQUIRED) else pd.Series(False,index=d.index)
coverage=float(ready.mean()) if len(d) else 0
print("[COVERAGE]",int(ready.sum()),"/",len(d),f"({coverage:.3%})",missing)

# STRICT: no imputation; validation set is only complete rows.
x=d.loc[ready].copy()
if len(x)==0: raise RuntimeError("No complete revalidation rows.")

# Return contract: use existing frozen/reconstructed fields only. Never invent exit semantics.
pairs=[
 ("net_return","reconstructed_fixed4_net_return"),
 ("net_return","fixed4_net_return"),
 ("gross_return","reconstructed_fixed4_raw_return"),
 ("gross_return","fixed4_raw_return"),
]
ret_pair=next(((a,b) for a,b in pairs if a in x.columns and b in x.columns),None)
if ret_pair is None:
 raise RuntimeError("Frozen candidate-vs-FIXED_4 return columns not found; do not reconstruct semantics silently.")
cand_col,base_col=ret_pair
x[cand_col]=pd.to_numeric(x[cand_col],errors="coerce")
x[base_col]=pd.to_numeric(x[base_col],errors="coerce")
v=x[[cand_col,base_col]].notna().all(axis=1)
x=x.loc[v].copy()
x["paired_diff"]=x[cand_col]-x[base_col]
print("[RETURN CONTRACT]",cand_col,"vs",base_col,"valid_pairs=",len(x))

# Fold consistency: positive mean paired advantage, no ranking/tuning.
fold_col="fold" if "fold" in x.columns else None
if fold_col:
 fold=x.groupby(fold_col).agg(n=("paired_diff","size"),candidate_mean=(cand_col,"mean"),fixed4_mean=(base_col,"mean"),paired_mean=("paired_diff","mean")).reset_index()
 fold["positive"]=fold["paired_mean"]>0
 positive_folds=int(fold["positive"].sum())
 total_folds=int(len(fold))
else:
 fold=pd.DataFrame()
 positive_folds=0; total_folds=0
fold.to_csv(OUT/"fold_consistency.csv",index=False)

# Paired bootstrap + block bootstrap by fold. This estimates uncertainty only; it does not tune policy.
rng=np.random.default_rng(SEED)
arr=x["paired_diff"].to_numpy(float)
boot=np.empty(N_BOOT)
for i in range(N_BOOT):
 boot[i]=rng.choice(arr,size=len(arr),replace=True).mean()
ci=np.quantile(boot,[.025,.975])
p_one=float((boot<=0).mean())

block=np.empty(N_BOOT)
if fold_col and total_folds:
 groups=[g["paired_diff"].to_numpy(float) for _,g in x.groupby(fold_col)]
 for i in range(N_BOOT):
  chosen=rng.integers(0,len(groups),size=len(groups))
  vals=np.concatenate([groups[j] for j in chosen])
  block[i]=vals.mean()
 block_ci=np.quantile(block,[.025,.975]); block_p=float((block<=0).mean())
else:
 block[:]=np.nan; block_ci=[np.nan,np.nan]; block_p=np.nan

# Sequential trade-level equity/MDD, descriptive only.
def equity_stats(r):
 r=np.asarray(r,float); eq=np.cumprod(1+r)
 peak=np.maximum.accumulate(np.r_[1.0,eq])[1:]
 dd=eq/peak-1
 return {"total_return":float(eq[-1]-1),"mdd":float(dd.min()),"mean_trade":float(np.mean(r)),"median_trade":float(np.median(r)),"win_rate":float(np.mean(r>0))}
cand_stats=equity_stats(x[cand_col]); base_stats=equity_stats(x[base_col])

# Exit attribution, descriptive; preserve frozen labels.
exit_attr=pd.DataFrame()
if "exit_reason" in x.columns:
 exit_attr=x.groupby("exit_reason",dropna=False).agg(n=("paired_diff","size"),candidate_mean=(cand_col,"mean"),fixed4_mean=(base_col,"mean"),paired_mean=("paired_diff","mean")).reset_index().sort_values("n",ascending=False)
 exit_attr.to_csv(OUT/"exit_attribution.csv",index=False)

# Gates. Coverage must be exactly complete; performance gates use predeclared 3/4 and positive uncertainty lower bounds.
coverage_pass=(len(d)>0 and int(ready.sum())==len(d))
fold_pass=(total_folds>=4 and positive_folds>=3)
bootstrap_pass=(float(ci[0])>0 and p_one<0.05)
block_pass=(np.isfinite(block_ci[0]) and float(block_ci[0])>0 and block_p<0.05)
validation_pass=bool(coverage_pass and fold_pass and bootstrap_pass and block_pass)

report={
 "schema":"kalman-open-revalidation-frozen-validation-v1.9",
 "research_only":True,"production_changed":False,"live_trading":False,"neon_write":False,
 "source_audit":str(AUDIT),"source_sha256_rows":src_hash,
 "rows":int(len(d)),"complete_rows":int(ready.sum()),"coverage":coverage,"missing":missing,
 "return_contract":{"candidate":cand_col,"baseline":base_col,"valid_pairs":int(len(x))},
 "fold":{"positive":positive_folds,"total":total_folds,"pass":bool(fold_pass)},
 "paired_bootstrap":{"n":N_BOOT,"mean_diff":float(arr.mean()),"ci95":[float(ci[0]),float(ci[1])],"p_one_sided":p_one,"pass":bool(bootstrap_pass)},
 "fold_block_bootstrap":{"n":N_BOOT,"ci95":[float(block_ci[0]),float(block_ci[1])],"p_one_sided":float(block_p),"pass":bool(block_pass)},
 "candidate":cand_stats,"fixed4":base_stats,
 "gates":{"coverage_pass":bool(coverage_pass),"fold_pass":bool(fold_pass),"paired_bootstrap_pass":bool(bootstrap_pass),"block_bootstrap_pass":bool(block_pass),"validation_pass":validation_pass},
 "promotion":{"eligible_now":False,"reason":"Prospective frozen evidence remains required even if retrospective revalidation passes."}
}
(OUT/"validation_report.json").write_text(json.dumps(report,indent=2,ensure_ascii=False))
pd.DataFrame({"paired_bootstrap_mean":boot}).to_parquet(OUT/"paired_bootstrap.parquet",index=False)
pd.DataFrame({"fold_block_bootstrap_mean":block}).to_parquet(OUT/"fold_block_bootstrap.parquet",index=False)

print("\n"+"="*90)
print("OPEN REVALIDATION FROZEN VALIDATION v1.9")
print("="*90)
print(json.dumps(report,indent=2,ensure_ascii=False))
if len(fold): print("\n[FOLDS]\n",fold.to_string(index=False))
if len(exit_attr): print("\n[EXIT ATTRIBUTION]\n",exit_attr.to_string(index=False))
print("\nOUTPUT:",OUT)
print("SAFETY: research_only=true | production_changed=false | live_trading=false | neon_write=false")
print("NEXT:", "PROSPECTIVE FROZEN GATE" if validation_pass else "KEEP R9 BLOCKED; inspect failed gate without retuning")
